In [1]:
import pandas as pd

results = pd.read_csv(
    "../data/raw/results.csv"
)

results["date"] = pd.to_datetime(
    results["date"]
)

results = results.dropna(
    subset=["home_score","away_score"]
)

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [2]:
results_2021 = results[
    results["date"] >= "2021-01-01"
]

results_2021.shape

(5678, 9)

In [3]:
results_2021["date"].min(), results_2021["date"].max()

(Timestamp('2021-01-12 00:00:00'), Timestamp('2026-06-09 00:00:00'))

In [4]:
len(results_2021)

5678

In [5]:
def match_result(row):

    if row["home_score"] > row["away_score"]:
        return "Home Win"

    elif row["home_score"] < row["away_score"]:
        return "Away Win"

    else:
        return "Draw"


results_2021["result"] = results_2021.apply(
    match_result,
    axis=1
)

In [6]:
home_matches = (
    results_2021
    .groupby("home_team")
    .size()
)

away_matches = (
    results_2021
    .groupby("away_team")
    .size()
)

matches_played = home_matches.add(
    away_matches,
    fill_value=0
)

modern_stats = pd.DataFrame({
    "matches_played": matches_played
})

In [7]:
home_wins = (
    results_2021[
        results_2021["result"] == "Home Win"
    ]
    .groupby("home_team")
    .size()
)

away_wins = (
    results_2021[
        results_2021["result"] == "Away Win"
    ]
    .groupby("away_team")
    .size()
)

wins = home_wins.add(
    away_wins,
    fill_value=0
)

modern_stats["wins"] = wins
modern_stats.fillna(0, inplace=True)

,matches_played,wins
home_team,,
Afghanistan,31.0,6.0
Albania,59.0,22.0
Alderney,2.0,0.0
Algeria,80.0,54.0
American Samoa,7.0,0.0
...,...,...
Yoruba Nation,2.0,0.0
Zambia,60.0,23.0
Zanzibar,4.0,3.0


In [8]:
modern_stats["win_rate"] = (
    modern_stats["wins"]
    / modern_stats["matches_played"]
)

In [9]:
home_goals = (
    results_2021
    .groupby("home_team")
    ["home_score"]
    .sum()
)

away_goals = (
    results_2021
    .groupby("away_team")
    ["away_score"]
    .sum()
)

modern_stats["goals_for"] = (
    home_goals.add(
        away_goals,
        fill_value=0
    )
)

In [10]:
home_ga = (
    results_2021
    .groupby("home_team")
    ["away_score"]
    .sum()
)

away_ga = (
    results_2021
    .groupby("away_team")
    ["home_score"]
    .sum()
)

modern_stats["goals_against"] = (
    home_ga.add(
        away_ga,
        fill_value=0
    )
)

modern_stats["goal_difference"] = (
    modern_stats["goals_for"]
    - modern_stats["goals_against"]
)

In [11]:
modern_stats[
    modern_stats["matches_played"] >= 20
].sort_values(
    "win_rate",
    ascending=False
).head(20)

,matches_played,wins,win_rate,goals_for,goals_against,goal_difference
home_team,,,,,,
Argentina,71.0,54.0,0.760563,152.0,31.0,121.0
Morocco,85.0,62.0,0.729412,172.0,38.0,134.0
Japan,68.0,48.0,0.705882,185.0,44.0,141.0
Iran,62.0,43.0,0.693548,131.0,46.0,85.0
Algeria,80.0,54.0,0.675000,180.0,58.0,122.0
Spain,72.0,47.0,0.652778,170.0,58.0,112.0
England,72.0,47.0,0.652778,157.0,43.0,114.0
Portugal,69.0,45.0,0.652174,166.0,56.0,110.0
Senegal,81.0,51.0,0.629630,146.0,55.0,91.0


In [12]:
modern_stats.to_excel(
    "../data/processed/modern_team_stats.xlsx"
)

In [13]:
results_2021.shape

(5678, 10)

In [14]:
results_2021["date"].min(), results_2021["date"].max()

(Timestamp('2021-01-12 00:00:00'), Timestamp('2026-06-09 00:00:00'))

In [15]:
modern_stats[
    modern_stats["matches_played"] >= 20
].sort_values(
    "win_rate",
    ascending=False
).head(20)

,matches_played,wins,win_rate,goals_for,goals_against,goal_difference
home_team,,,,,,
Argentina,71.0,54.0,0.760563,152.0,31.0,121.0
Morocco,85.0,62.0,0.729412,172.0,38.0,134.0
Japan,68.0,48.0,0.705882,185.0,44.0,141.0
Iran,62.0,43.0,0.693548,131.0,46.0,85.0
Algeria,80.0,54.0,0.675000,180.0,58.0,122.0
Spain,72.0,47.0,0.652778,170.0,58.0,112.0
England,72.0,47.0,0.652778,157.0,43.0,114.0
Portugal,69.0,45.0,0.652174,166.0,56.0,110.0
Senegal,81.0,51.0,0.629630,146.0,55.0,91.0


In [16]:
modern_stats.to_excel(
    "../data/processed/modern_team_stats.xlsx"
)

In [18]:
draws = (
    results_2021[
        results_2021["result"] == "Draw"
    ]
    .groupby("home_team")
    .size()
)

modern_stats["draws"] = draws

modern_stats["draws"] = (
    modern_stats["draws"]
    .fillna(0)
)

In [19]:
modern_stats["losses"] = (
    modern_stats["matches_played"]
    - modern_stats["wins"]
    - modern_stats["draws"]
)

In [20]:
modern_stats["points"] = (
    modern_stats["wins"] * 3
    + modern_stats["draws"]
)

In [21]:
modern_stats["points_per_match"] = (
    modern_stats["points"]
    / modern_stats["matches_played"]
)

In [22]:
modern_stats.sort_values(
    "points_per_match",
    ascending=False
).head(20)

,matches_played,wins,win_rate,goals_for,goals_against,goal_difference,draws,losses,points,points_per_match
home_team,,,,,,,,,,
Tamil Eelam,6.0,6.0,1.000000,20.0,3.0,17.0,0.0,0.0,18.0,3.000000
Székely Land,2.0,2.0,1.000000,10.0,2.0,8.0,0.0,0.0,6.0,3.000000
Maule Sur,2.0,2.0,1.000000,2.0,0.0,2.0,0.0,0.0,6.0,3.000000
Kernow,1.0,1.0,1.000000,2.0,1.0,1.0,0.0,0.0,3.0,3.000000
Elba Island,1.0,1.0,1.000000,5.0,0.0,5.0,0.0,0.0,3.0,3.000000
Jersey,11.0,9.0,0.818182,33.0,11.0,22.0,1.0,1.0,28.0,2.545455
Ynys Môn,6.0,5.0,0.833333,13.0,7.0,6.0,0.0,1.0,15.0,2.500000
Argentina,71.0,54.0,0.760563,152.0,31.0,121.0,7.0,10.0,169.0,2.380282
Morocco,85.0,62.0,0.729412,172.0,38.0,134.0,11.0,12.0,197.0,2.317647


In [23]:
results_2021["tournament"].value_counts()

tournament
FIFA World Cup qualification            1610
Friendly                                1484
UEFA Nations League                      354
African Cup of Nations qualification     341
CONCACAF Nations League                  320
                                        ... 
ASEAN Championship qualification           2
Mukuru 4 Nations                           2
Tri-Nations Cup                            2
CONMEBOL–UEFA Cup of Champions             1
South Asian Super Cup                      1
Name: count, Length: 66, dtype: int64

In [24]:
def get_tournament_weight(tournament):

    tournament = str(tournament)

    # Mundial
    if tournament == "FIFA World Cup":
        return 5

    # Copas continentales principales
    elif (
        "Copa América" in tournament
        or "UEFA Euro" in tournament
        or "African Cup of Nations" == tournament
        or "AFC Asian Cup" == tournament
        or "CONCACAF Gold Cup" == tournament
    ):
        return 4

    # Eliminatorias
    elif "qualification" in tournament:
        return 3

    # Nations League
    elif "Nations League" in tournament:
        return 2

    # Amistosos
    elif tournament == "Friendly":
        return 1

    # Otros torneos
    else:
        return 2

In [25]:
results_2021["competition_weight"] = (
    results_2021["tournament"]
    .apply(get_tournament_weight)
)

results_2021[
    [
        "tournament",
        "competition_weight"
    ]
].head(20)

,tournament,competition_weight
43722,Friendly,1
43723,Friendly,1
43724,Friendly,1
43725,Friendly,1
43726,Friendly,1
43727,Friendly,1
43728,Friendly,1
43729,Friendly,1
43730,Friendly,1
43731,Friendly,1


In [26]:
def get_match_points(row):

    if row["result"] == "Draw":
        return 1

    elif (
        row["result"] == "Home Win"
    ):
        return 3

    elif (
        row["result"] == "Away Win"
    ):
        return 3

    return 0

In [27]:
results_2021["match_points"] = (
    results_2021.apply(
        get_match_points,
        axis=1
    )
)

In [28]:
results_2021["weighted_points"] = (
    results_2021["match_points"]
    * results_2021["competition_weight"]
)

In [29]:
home_weighted = (
    results_2021
    .groupby("home_team")
    ["weighted_points"]
    .sum()
)

away_weighted = (
    results_2021
    .groupby("away_team")
    ["weighted_points"]
    .sum()
)

weighted_score = (
    home_weighted
    .add(
        away_weighted,
        fill_value=0
    )
)

modern_stats["weighted_score"] = (
    weighted_score
)

In [30]:
modern_stats["weighted_score_per_match"] = (
    modern_stats["weighted_score"]
    / modern_stats["matches_played"]
)

In [31]:
modern_stats[
    modern_stats["matches_played"] >= 20
].sort_values(
    "weighted_score_per_match",
    ascending=False
).head(20)

,matches_played,wins,win_rate,goals_for,goals_against,goal_difference,draws,losses,points,points_per_match,weighted_score,weighted_score_per_match
home_team,,,,,,,,,,,,
São Tomé and Príncipe,22.0,1.0,0.045455,13.0,64.0,-51.0,1.0,20.0,4.0,0.181818,174.0,7.909091
Djibouti,26.0,3.0,0.115385,19.0,74.0,-55.0,1.0,22.0,10.0,0.384615,202.0,7.769231
Portugal,69.0,45.0,0.652174,166.0,56.0,110.0,5.0,19.0,140.0,2.028986,520.0,7.536232
Denmark,67.0,38.0,0.567164,129.0,60.0,69.0,6.0,23.0,120.0,1.791045,493.0,7.358209
Netherlands,69.0,42.0,0.608696,169.0,70.0,99.0,10.0,17.0,136.0,1.971014,503.0,7.289855
Argentina,71.0,54.0,0.760563,152.0,31.0,121.0,7.0,10.0,169.0,2.380282,517.0,7.281690
England,72.0,47.0,0.652778,157.0,43.0,114.0,10.0,15.0,151.0,2.097222,524.0,7.277778
Taiwan,28.0,6.0,0.214286,34.0,68.0,-34.0,2.0,20.0,20.0,0.714286,200.0,7.142857
Cameroon,64.0,29.0,0.453125,87.0,55.0,32.0,11.0,24.0,98.0,1.531250,454.0,7.093750


In [32]:
home_df = results_2021.copy()

home_df["team"] = home_df["home_team"]
home_df["opponent"] = home_df["away_team"]

home_df["goals_for"] = home_df["home_score"]
home_df["goals_against"] = home_df["away_score"]

In [33]:
home_df["points"] = 0

home_df.loc[
    home_df["home_score"] >
    home_df["away_score"],
    "points"
] = 3

home_df.loc[
    home_df["home_score"] ==
    home_df["away_score"],
    "points"
] = 1

In [34]:
away_df = results_2021.copy()

away_df["team"] = away_df["away_team"]
away_df["opponent"] = away_df["home_team"]

away_df["goals_for"] = away_df["away_score"]
away_df["goals_against"] = away_df["home_score"]

In [35]:
away_df["points"] = 0

away_df.loc[
    away_df["away_score"] >
    away_df["home_score"],
    "points"
] = 3

away_df.loc[
    away_df["away_score"] ==
    away_df["home_score"],
    "points"
] = 1

In [36]:
home_df = home_df[
    [
        "date",
        "team",
        "opponent",
        "tournament",
        "goals_for",
        "goals_against",
        "points"
    ]
]

away_df = away_df[
    [
        "date",
        "team",
        "opponent",
        "tournament",
        "goals_for",
        "goals_against",
        "points"
    ]
]

In [37]:
team_matches = pd.concat(
    [
        home_df,
        away_df
    ],
    ignore_index=True
)

In [38]:
team_matches.shape

(11356, 7)

In [39]:
team_matches.head(20)

,date,team,opponent,tournament,goals_for,goals_against,points
0,2021-01-12,United Arab Emirates,Iraq,Friendly,0.0,0.0,1
1,2021-01-18,Kuwait,Palestine,Friendly,0.0,1.0,0
2,2021-01-19,Dominican Republic,Puerto Rico,Friendly,0.0,1.0,0
3,2021-01-22,Guatemala,Puerto Rico,Friendly,1.0,0.0,3
4,2021-01-25,Dominican Republic,Serbia,Friendly,0.0,0.0,1
5,2021-01-27,Iraq,Kuwait,Friendly,2.0,1.0,3
6,2021-01-28,Panama,Serbia,Friendly,0.0,0.0,1
7,2021-01-31,United States,Trinidad and Tobago,Friendly,7.0,0.0,3
8,2021-02-01,Jordan,Tajikistan,Friendly,2.0,0.0,3
9,2021-02-05,Jordan,Tajikistan,Friendly,0.0,1.0,0


In [40]:
team_matches.head()

,date,team,opponent,tournament,goals_for,goals_against,points
0,2021-01-12,United Arab Emirates,Iraq,Friendly,0.0,0.0,1
1,2021-01-18,Kuwait,Palestine,Friendly,0.0,1.0,0
2,2021-01-19,Dominican Republic,Puerto Rico,Friendly,0.0,1.0,0
3,2021-01-22,Guatemala,Puerto Rico,Friendly,1.0,0.0,3
4,2021-01-25,Dominican Republic,Serbia,Friendly,0.0,0.0,1


In [41]:
def get_tournament_weight(tournament):

    tournament = str(tournament)

    if tournament == "FIFA World Cup":
        return 5

    elif (
        "Copa América" in tournament
        or "UEFA Euro" in tournament
        or tournament == "African Cup of Nations"
        or tournament == "AFC Asian Cup"
        or tournament == "CONCACAF Gold Cup"
    ):
        return 4

    elif "qualification" in tournament:
        return 3

    elif "Nations League" in tournament:
        return 2

    elif tournament == "Friendly":
        return 1

    else:
        return 2

In [42]:
team_matches["competition_weight"] = (
    team_matches["tournament"]
    .apply(get_tournament_weight)
)

In [43]:
team_matches["weighted_points"] = (
    team_matches["points"]
    * team_matches["competition_weight"]
)

In [44]:
team_matches.head()

,date,team,opponent,tournament,goals_for,goals_against,points,competition_weight,weighted_points
0,2021-01-12,United Arab Emirates,Iraq,Friendly,0.0,0.0,1,1,1
1,2021-01-18,Kuwait,Palestine,Friendly,0.0,1.0,0,1,0
2,2021-01-19,Dominican Republic,Puerto Rico,Friendly,0.0,1.0,0,1,0
3,2021-01-22,Guatemala,Puerto Rico,Friendly,1.0,0.0,3,1,3
4,2021-01-25,Dominican Republic,Serbia,Friendly,0.0,0.0,1,1,1


In [45]:
team_summary = (
    team_matches
    .groupby("team")
    .agg(
        matches_played=("team","count"),
        total_points=("points","sum"),
        weighted_points=("weighted_points","sum"),
        goals_for=("goals_for","sum"),
        goals_against=("goals_against","sum")
    )
)

In [46]:
team_summary["points_per_match"] = (
    team_summary["total_points"]
    / team_summary["matches_played"]
)

team_summary["weighted_points_per_match"] = (
    team_summary["weighted_points"]
    / team_summary["matches_played"]
)

team_summary["goal_difference"] = (
    team_summary["goals_for"]
    - team_summary["goals_against"]
)

In [47]:
team_summary[
    team_summary["matches_played"] >= 20
].sort_values(
    "weighted_points_per_match",
    ascending=False
).head(20)

,matches_played,total_points,weighted_points,goals_for,goals_against,points_per_match,weighted_points_per_match,goal_difference
team,,,,,,,,
Argentina,71,174,466,152.0,31.0,2.450704,6.563380,121.0
England,72,156,461,157.0,43.0,2.166667,6.402778,114.0
Morocco,85,203,529,172.0,38.0,2.388235,6.223529,134.0
Spain,72,160,446,170.0,58.0,2.222222,6.194444,112.0
Senegal,81,174,484,146.0,55.0,2.148148,5.975309,91.0
Portugal,69,147,412,166.0,56.0,2.130435,5.971014,110.0
Netherlands,69,142,410,169.0,70.0,2.057971,5.942029,99.0
France,71,148,421,153.0,63.0,2.084507,5.929577,90.0
Iran,62,139,349,131.0,46.0,2.241935,5.629032,85.0


In [48]:
team_summary["gd_per_match"] = (
    team_summary["goal_difference"]
    / team_summary["matches_played"]
)

In [49]:
team_summary[
    team_summary["matches_played"] >= 20
].sort_values(
    "gd_per_match",
    ascending=False
).head(20)

,matches_played,total_points,weighted_points,goals_for,goals_against,points_per_match,weighted_points_per_match,goal_difference,gd_per_match
team,,,,,,,,,
Japan,68,153,379,185.0,44.0,2.250000,5.573529,141.0,2.073529
Argentina,71,174,466,152.0,31.0,2.450704,6.563380,121.0,1.704225
Portugal,69,147,412,166.0,56.0,2.130435,5.971014,110.0,1.594203
England,72,156,461,157.0,43.0,2.166667,6.402778,114.0,1.583333
Morocco,85,203,529,172.0,38.0,2.388235,6.223529,134.0,1.576471
Spain,72,160,446,170.0,58.0,2.222222,6.194444,112.0,1.555556
Algeria,80,180,407,180.0,58.0,2.250000,5.087500,122.0,1.525000
Netherlands,69,142,410,169.0,70.0,2.057971,5.942029,99.0,1.434783
Iran,62,139,349,131.0,46.0,2.241935,5.629032,85.0,1.370968


In [50]:
team_summary["power_score_v1"] = (
    team_summary["weighted_points_per_match"] * 0.60
    +
    team_summary["gd_per_match"] * 0.40
)

In [51]:
team_summary[
    team_summary["matches_played"] >= 20
].sort_values(
    "power_score_v1",
    ascending=False
).head(20)

,matches_played,total_points,weighted_points,goals_for,goals_against,points_per_match,weighted_points_per_match,goal_difference,gd_per_match,power_score_v1
team,,,,,,,,,,
Argentina,71,174,466,152.0,31.0,2.450704,6.563380,121.0,1.704225,4.619718
England,72,156,461,157.0,43.0,2.166667,6.402778,114.0,1.583333,4.475000
Morocco,85,203,529,172.0,38.0,2.388235,6.223529,134.0,1.576471,4.364706
Spain,72,160,446,170.0,58.0,2.222222,6.194444,112.0,1.555556,4.338889
Portugal,69,147,412,166.0,56.0,2.130435,5.971014,110.0,1.594203,4.220290
Japan,68,153,379,185.0,44.0,2.250000,5.573529,141.0,2.073529,4.173529
Netherlands,69,142,410,169.0,70.0,2.057971,5.942029,99.0,1.434783,4.139130
France,71,148,421,153.0,63.0,2.084507,5.929577,90.0,1.267606,4.064789
Senegal,81,174,484,146.0,55.0,2.148148,5.975309,91.0,1.123457,4.034568


In [52]:
team_summary["power_score_v1"] = (
    team_summary["weighted_points_per_match"] * 0.60
    +
    team_summary["gd_per_match"] * 0.40
)

In [53]:
team_summary[
    team_summary["matches_played"] >= 20
].sort_values(
    "power_score_v1",
    ascending=False
).head(20)

,matches_played,total_points,weighted_points,goals_for,goals_against,points_per_match,weighted_points_per_match,goal_difference,gd_per_match,power_score_v1
team,,,,,,,,,,
Argentina,71,174,466,152.0,31.0,2.450704,6.563380,121.0,1.704225,4.619718
England,72,156,461,157.0,43.0,2.166667,6.402778,114.0,1.583333,4.475000
Morocco,85,203,529,172.0,38.0,2.388235,6.223529,134.0,1.576471,4.364706
Spain,72,160,446,170.0,58.0,2.222222,6.194444,112.0,1.555556,4.338889
Portugal,69,147,412,166.0,56.0,2.130435,5.971014,110.0,1.594203,4.220290
Japan,68,153,379,185.0,44.0,2.250000,5.573529,141.0,2.073529,4.173529
Netherlands,69,142,410,169.0,70.0,2.057971,5.942029,99.0,1.434783,4.139130
France,71,148,421,153.0,63.0,2.084507,5.929577,90.0,1.267606,4.064789
Senegal,81,174,484,146.0,55.0,2.148148,5.975309,91.0,1.123457,4.034568


In [54]:
team_summary.to_excel(
    "../data/processed/modern_team_stats_v2.xlsx"
)

In [55]:
team_summary.to_excel(
    "../data/processed/modern_team_stats_v2.xlsx"
)

In [56]:
import pandas as pd

elo = pd.read_csv(
    "../data/raw/eloratings.csv"
)

elo.head()

,date,team,rating,change
0,1872-11-30,England,2003.0,3
1,1872-11-30,Scotland,1997.0,-3
2,1873-03-08,England,2014.0,11
3,1873-03-08,Scotland,1986.0,-11
4,1874-03-07,England,2006.0,-8


In [57]:
elo.columns

Index(['date', 'team', 'rating', 'change'], dtype='str')

In [58]:
elo.shape

(6678, 4)

In [59]:
elo["date"] = pd.to_datetime(
    elo["date"]
)

ValueError: time data "3/23/1901" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [60]:
elo["date"] = pd.to_datetime(
    elo["date"],
    format="mixed"
)

In [61]:
elo = elo.sort_values("date")

latest_elo = (
    elo.groupby("team")
       .tail(1)
)

latest_elo.sort_values(
    "rating",
    ascending=False
).head(20)

,date,team,rating,change
6434,2025-12-13,Spain,2171.0,0
607,1974-09-04,West Germany,2144.0,2
6435,2025-12-13,Argentina,2113.0,0
6436,2025-12-13,France,2062.0,0
6437,2025-12-13,England,2042.0,0
6438,2025-12-13,Colombia,1998.0,0
6439,2025-12-13,Brazil,1979.0,0
6440,2025-12-13,Portugal,1976.0,0
6441,2025-12-13,Netherlands,1959.0,0
6443,2025-12-13,Croatia,1933.0,0


In [62]:
latest_elo = latest_elo[
    [
        "team",
        "rating"
    ]
]

In [63]:
team_summary = (
    team_summary
    .reset_index()
)

In [64]:
team_summary = team_summary.merge(
    latest_elo,
    on="team",
    how="left"
)

In [65]:
team_summary.head()

,team,matches_played,total_points,weighted_points,goals_for,goals_against,points_per_match,weighted_points_per_match,goal_difference,gd_per_match,power_score_v1,rating
0,Afghanistan,31,28,63,24.0,51.0,0.903226,2.032258,-27.0,-0.870968,0.870968,1033.0
1,Albania,59,79,200,62.0,60.0,1.338983,3.389831,2.0,0.033898,2.047458,1664.0
2,Alderney,2,0,0,1.0,8.0,0.000000,0.000000,-7.0,-3.500000,-1.400000,NaN
3,Algeria,80,180,407,180.0,58.0,2.250000,5.087500,122.0,1.525000,3.662500,1726.0
4,American Samoa,7,0,0,4.0,44.0,0.000000,0.000000,-40.0,-5.714286,-2.285714,NaN


In [66]:
team_summary[
    team_summary["rating"].isna()
][["team"]]

,team
2,Alderney
4,American Samoa
8,Antigua and Barbuda
14,Aymara
20,Basque Country
...,...
252,West Papua
253,Western Isles
255,Ynys Môn
256,Yoruba Nation


In [67]:
team_summary["rating"].notna().sum()

np.int64(173)

In [68]:
team_summary.shape

(261, 12)

In [69]:
model_df = team_summary[
    team_summary["rating"].notna()
].copy()

In [70]:
from sklearn.preprocessing import MinMaxScaler